In [ ]:
!pip install plotly

In [ ]:
!pip install plotly


import pandas as pd
import plotly.express as px


df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/OSCARS_con_streaming-7.csv", sep=";")


df['Tipo'] = df['es_streaming'].map({1: 'Streaming', 0: 'Tradicional'})


nominaciones = df.groupby(['Year', 'Tipo']).size().reset_index(name='Nominaciones')


victorias = df[df['winner'] == 1].groupby(['Year', 'Tipo']).size().reset_index(name='Victorias')


df_linea = pd.merge(nominaciones, victorias, on=['Year', 'Tipo'], how='left').fillna(0)

line_dash_map={
    'Nominaciones': 'dot',
    'Victorias': 'solid'
}


df_long = pd.melt(
    df_linea,
    id_vars=['Year', 'Tipo'],
    value_vars=['Nominaciones', 'Victorias'],
    var_name='Métrica',
    value_name='Total'
)


fig = px.line(
    df_long,
    x='Year',
    y='Total',
    color='Tipo',
    line_dash='Métrica',
    markers=True,
    title='Evolución de nominaciones y victorias: Streaming vs Tradicional',
    color_discrete_map={
        'Streaming': '#800020',
        'Tradicional': '#C5A300'
    }
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis_title='Cantidad',
    xaxis_title='Año',
    legend_title='Tipo de Distribución'
)


fig.show()

fig.write_html("/content/nominaciones_vs_victorias_streaming.html")

In [ ]:
import pandas as pd
import plotly.express as px

# Cargar datos
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/OSCARS_con_streaming-7.csv", sep=";")

df['Tipo'] = df['es_streaming'].map({1: 'Streaming', 0: 'Tradicional'})

nominaciones = df.groupby(['Year', 'Tipo']).size().reset_index(name='Nominaciones')
victorias = df[df['winner'] == 1].groupby(['Year', 'Tipo']).size().reset_index(name='Victorias')

df_linea = pd.merge(nominaciones, victorias, on=['Year', 'Tipo'], how='left').fillna(0)

df_long = pd.melt(
    df_linea,
    id_vars=['Year', 'Tipo'],
    value_vars=['Nominaciones', 'Victorias'],
    var_name='Métrica',
    value_name='Total'
)

fig = px.line(
    df_long,
    x='Year',
    y='Total',
    color='Tipo',
    line_dash='Métrica',
    line_dash_map={
        'Nominaciones': 'dot',   # Punteada
        'Victorias': 'solid'     # Sólida
    },
    markers=True,
    title='Evolución de nominaciones y victorias: Streaming vs Tradicional',
    color_discrete_map={
        'Streaming': '#800020',
        'Tradicional': '#C5A300'
    }
)

fig.update_layout(
    plot_bgcolor='white',
    yaxis_title='Cantidad',
    xaxis_title='Año',
    legend_title='Tipo de Distribución'
)

fig.show()

fig.write_html("/content/nominaciones_vs_victorias_streaming.html")
from google.colab import files
files.download("/content/nominaciones_vs_victorias_streaming.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import json

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/OSCARS_con_streaming-7.csv", sep=";")

# 2. Definir plataformas de streaming
streaming = ['Netflix', 'Amazon', 'Hulu', 'Disney+']

# 3. Filtrar filas donde alguna de las columnas dist_1 a dist_5 sea de streaming
columnas_distribuidoras = ['dist_1', 'dist_2', 'dist_3', 'dist_4', 'dist_5']
df_streaming = df[
    df[columnas_distribuidoras].isin(streaming).any(axis=1)
]

# 4. Crear columna con la primera distribuidora de streaming encontrada
def detectar_streaming(row):
    for col in columnas_distribuidoras:
        if row[col] in streaming:
            return row[col]
    return None

df_streaming['Distribuidora'] = df_streaming.apply(detectar_streaming, axis=1)

# 5. Crear columnas para visualización
df_streaming['Nominada'] = 1  # Todas están nominadas por estar en la base
df_streaming['Ganadora'] = df_streaming['winner']  # Ya existe como 1 o 0

# 6. Seleccionar columnas necesarias
df_plot = df_streaming[['Year', 'Film', 'Distribuidora', 'Nominada', 'Ganadora']]

# 7. Exportar a JSON
json_data = df_plot.to_dict(orient='records')
with open('/content/datos_streaming.json', 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=2)

# 8. Mostrar bloque JS para pegar en HTML
print("🔁 Copia esto en tu HTML:\n")
print("const data = ", json.dumps(json_data, ensure_ascii=False, indent=2), ";")


🔁 Copia esto en tu HTML:

const data =  [] ;


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/OSCARS_con_streaming-7.csv", sep=";")

# 2. Definir plataformas de streaming
plataformas = {
    'netflix': 'Netflix',
    'amazon studios': 'Amazon',
    'prime video': 'Amazon',
    'apple': 'Apple',
}


# 3. Usar solo dist_1 y dist_2
cols_dist = ['dist_1', 'dist_2']

# 4. Detectar distribuidora de streaming
def detectar_streaming(row):
    for col in cols_dist:
        val = str(row[col]).lower()
        for key in plataformas:
            if key in val:
                return plataformas[key]
    return None

df['Distribuidora'] = df.apply(detectar_streaming, axis=1)
df_stream = df[df['Distribuidora'].notnull()].copy()

# 5. Agrupar por Year + Category + Distribuidora → contar nominaciones
agrupado = (
    df_stream.groupby(['Year', 'Category', 'Distribuidora'])
    .size()
    .reset_index(name='nominaciones')
)

# 6. Agregar jitter para separar los puntos verticalmente
np.random.seed(42)
agrupado['jitter'] = agrupado['Category'].apply(lambda x: np.random.uniform(-0.3, 0.3))

# 7. Crear gráfico beeswarm-like
fig = px.scatter(
    agrupado,
    x='Year',
    y='Category',
    size='nominaciones',
    color='Distribuidora',
    color_discrete_map={
        'Netflix': '#800020',
        'Amazon': '#9A6324',
        'Apple': '#228B22',
        'HBO': '#4682B4',
    },
    hover_data=['nominaciones'],
    title='🎬 Nominaciones por Categoría y Año (Solo dist_1 y dist_2)'
)

fig.update_layout(height=700)
fig.show()
fig.write_html("nominaciones_streaming.html")

from google.colab import files
files.download("nominaciones_streaming.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/OSCARS_con_streaming-7.csv", sep=";")

# 2. Diccionario de plataformas de streaming
plataformas = {
    'netflix': 'Netflix',
    'amazon studios': 'Amazon',
    'prime video': 'Amazon',
    'apple': 'Apple',
}

# 3. Solo usar dist_1 y dist_2
cols_dist = ['dist_1', 'dist_2']

# 4. Detectar plataforma de streaming
def detectar_streaming(row):
    for col in cols_dist:
        val = str(row[col]).lower()
        for key in plataformas:
            if key in val:
                return plataformas[key]
    return None

df['Distribuidora'] = df.apply(detectar_streaming, axis=1)
df_stream = df[df['Distribuidora'].notnull()].copy()

# 5. Filtrar solo ganadoras
df_ganadoras = df_stream[df_stream['winner'] == 1].copy()

# 6. Agrupar por Year + Category + Distribuidora para contar victorias
agrupado_g = (
    df_ganadoras.groupby(['Year', 'Category', 'Distribuidora'])
    .size()
    .reset_index(name='victorias')
)

# 7. Jitter para dispersión visual
np.random.seed(42)
agrupado_g['jitter'] = agrupado_g['Category'].apply(lambda x: np.random.uniform(-0.3, 0.3))

# 8. Crear gráfico
fig_g = px.scatter(
    agrupado_g,
    x='Year',
    y='Category',
    size='victorias',
    color='Distribuidora',
    color_discrete_map={
        'Netflix': '#800020',
        'Amazon': '#9A6324',
        'Apple': '#228B22',
        'Paramount': '#4682B4',
        'Disney+': '#FFD700',
        'HBO': '#663399'
    },
    hover_data=['victorias'],
    title='🏆 Victorias por Categoría y Año (Streaming - dist_1 y dist_2)'
)

fig_g.update_layout(height=700)
fig_g.show()
fig_g.write_html("victorias_streaming.html")

from google.colab import files
files.download("victorias_streaming.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Instalar altair si no está
!pip install altair --quiet

# Importar librerías
import pandas as pd
import altair as alt
alt.renderers.enable('colab')

# Datos originales
data = [
    {"Plataforma": "Netflix", "Evento": "Fundación", "Año": 1997},
    {"Plataforma": "Netflix", "Evento": "Inicio Streaming", "Año": 2007},
    {"Plataforma": "Netflix", "Evento": "Primer Original", "Año": 2013},
    {"Plataforma": "Netflix", "Evento": "1ª Nominación Oscar", "Año": 2014},
    {"Plataforma": "Netflix", "Evento": "1º Oscar Ganado", "Año": 2017},

    {"Plataforma": "Amazon Prime Video", "Evento": "Fundación", "Año": 2006},
    {"Plataforma": "Amazon Prime Video", "Evento": "Inicio Streaming", "Año": 2006},
    {"Plataforma": "Amazon Prime Video", "Evento": "Primer Original", "Año": 2013},
    {"Plataforma": "Amazon Prime Video", "Evento": "1ª Nominación Oscar", "Año": 2017},
    {"Plataforma": "Amazon Prime Video", "Evento": "1º Oscar Ganado", "Año": 2017},

    {"Plataforma": "Apple TV+", "Evento": "Fundación", "Año": 2019},
    {"Plataforma": "Apple TV+", "Evento": "Inicio Streaming", "Año": 2019},
    {"Plataforma": "Apple TV+", "Evento": "Primer Original", "Año": 2019},
    {"Plataforma": "Apple TV+", "Evento": "1ª Nominación Oscar", "Año": 2021},
    {"Plataforma": "Apple TV+", "Evento": "1º Oscar Ganado", "Año": 2022},

    {"Plataforma": "Disney+", "Evento": "Fundación", "Año": 2019},
    {"Plataforma": "Disney+", "Evento": "Inicio Streaming", "Año": 2019},
    {"Plataforma": "Disney+", "Evento": "Primer Original", "Año": 2019},
    {"Plataforma": "Disney+", "Evento": "1ª Nominación Oscar", "Año": 2023},

    {"Plataforma": "HBO Max", "Evento": "Fundación", "Año": 2020},
    {"Plataforma": "HBO Max", "Evento": "Inicio Streaming", "Año": 2020},
    {"Plataforma": "HBO Max", "Evento": "Primer Original", "Año": 2020},
    {"Plataforma": "HBO Max", "Evento": "1ª Nominación Oscar", "Año": 2021},

    {"Plataforma": "Hulu", "Evento": "Fundación", "Año": 2007},
    {"Plataforma": "Hulu", "Evento": "Inicio Streaming", "Año": 2008},
    {"Plataforma": "Hulu", "Evento": "Primer Original", "Año": 2011}
]

df = pd.DataFrame(data)

# Logos
logo_urls = {
    "Netflix": "https://images.ctfassets.net/4cd45et68cgf/Rx83JoRDMkYNlMC9MKzcB/2b14d5a59fc3937afd3f03191e19502d/Netflix-Symbol.png?w=700&h=456",
    "Amazon Prime Video": "https://upload.wikimedia.org/wikipedia/commons/thumb/c/ca/Amazon_Prime_Video_logo_%282024%29.svg/2048px-Amazon_Prime_Video_logo_%282024%29.svg.png",
    "Apple TV+": "https://upload.wikimedia.org/wikipedia/commons/thumb/a/ad/AppleTVLogo.svg/1200px-AppleTVLogo.svg.png",
    "Disney+": "https://upload.wikimedia.org/wikipedia/commons/f/fa/Disney_plus_icon.png",
    "HBO Max": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQOAv2FstT4qyGuhI0ZJo2UhsSxpZpVYMtakQ&s",
    "Hulu": "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRkluPp1QJ4oHhT2WPzDIB5Y2qc-E2jRtG4eDm8QF0C4aoiGhT1os37wuSCBbzTx3VJHjg&usqp=CAU"
}

df["Logo"] = df["Plataforma"].map(logo_urls)

# Colores por plataforma (opcional, pero mantenido)
platform_colors = {
    "Netflix": "#E50914",
    "Amazon Prime Video": "#00A8E1",
    "Apple TV+": "#A2AAAD",
    "Disney+": "#113CCF",
    "HBO Max": "#8A2BE2",
    "Hulu": "#1CE783"
}

# Barras blancas como estructura
bars = alt.Chart(df).mark_bar(
    color='white',
    stroke='lightgray',
    strokeWidth=0.5
).encode(
    x=alt.X('Año:O', axis=alt.Axis(labelAngle=0, title="Año")),
    y=alt.Y('Evento:N', sort='-x', title="Eventos"),
    tooltip=['Plataforma', 'Evento', 'Año']
)

# Logos más pequeños
logos = alt.Chart(df).mark_image(
    width=40,   # antes 40
    height=50
).encode(
    x=alt.X('Año:O'),
    y=alt.Y('Evento:N', sort='-x'),
    url='Logo:N'
)

# Combinado con dimensiones reducidas
chart = (bars + logos).properties(
    width=600,  # antes 900
    height=400, # antes 500
    title="Hitos Clave de Plataformas de Streaming en los Oscar"
).configure_axis(
    labelFontSize=11,
    titleFontSize=13
).configure_title(
    fontSize=18,
    anchor='start'
)
chart
# Guardar HTML
chart.save('/content/hitos_streaming_oscar.html')
from google.colab import files
files.download('/content/hitos_streaming_oscar.html')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import pandas as pd
import plotly.express as px

# Datos base
data = {
    "Empresa": [
        "Disney", "Comcast", "Google/YouTube",
        "Warner Bros. Discovery", "Netflix", "Paramount Global"
    ],
    "Inversión Total 2024 (USD bn)": [
        35.8, 24.5, 17.6, 16.8, 16.0, 15.1
    ],
    "Producción Original (USD bn)": [
        25.0, 15.0, 10.0, 12.0, 12.0, 10.0
    ],
    "Compra/Licencia (USD bn)": [
        10.8, 9.5, 7.6, 4.8, 4.0, 5.1
    ]
}

df = pd.DataFrame(data)

# Pasar a formato largo
df_long = df.melt(
    id_vars="Empresa",
    value_vars=["Producción Original (USD bn)", "Compra/Licencia (USD bn)"],
    var_name="Tipo",
    value_name="Inversión (USD bn)"
)

# Gráfico con dorado y burdeos
fig = px.bar(
    df_long,
    x="Empresa",
    y="Inversión (USD bn)",
    color="Tipo",
    text="Inversión (USD bn)",
    title="Inversión en Contenido 2024: Producción vs Licencia",
    color_discrete_map={
        "Producción Original (USD bn)": "#800020",  # Dorado
        "Compra/Licencia (USD bn)": "#EFBF04"       # Burdeos
    }
)

fig.update_traces(
    texttemplate='%{text:.1f}B',
    textposition='inside'
)

fig.update_layout(
    yaxis_title="Inversión (miles de millones USD)",
    xaxis_title="Empresa",
    barmode='stack',
    xaxis_tickangle=-30,
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

# Guardar como HTML interactivo
fig.write_html("grafico_streaming.html")




In [15]:
import pandas as pd
import plotly.express as px

# Datos estimados (en miles de millones de USD)
data = {
    "Empresa": [
        "Disney (Streaming)", "Comcast (Streaming)", "Google/YouTube",
        "Warner Bros. Discovery (Streaming)", "Netflix", "Paramount Global (Streaming)",
        "Disney (Tradicional)", "Comcast (Tradicional)",
        "Warner Bros. Discovery (Tradicional)", "Paramount (Tradicional)"
    ],
    "Tipo": [
        "Streaming", "Streaming", "Streaming", "Streaming",
        "Streaming", "Streaming",
        "Tradicional", "Tradicional", "Tradicional", "Tradicional"
    ],
    "Inversión 2024 (USD bn)": [
        35.8, 24.5, 17.6, 16.8, 16.0, 15.1,
        10.0, 5.0, 6.0, 3.0
    ]
}

df = pd.DataFrame(data)

# Ordenar empresas por inversión para que quede más claro
df = df.sort_values("Inversión 2024 (USD bn)", ascending=False)

# Gráfico interactivo de barras por empresa
fig = px.bar(
    df,
    x="Empresa",
    y="Inversión 2024 (USD bn)",
    color="Tipo",
    text="Inversión 2024 (USD bn)",
    color_discrete_map={"Streaming": "#800020", "Tradicional": "#EFBF04"},
    title="Inversión en Contenido 2024 por Empresa"
)

fig.update_traces(
    texttemplate='%{text:.1f}B',
    textposition='outside'
)

fig.update_layout(
    yaxis_title="Inversión (miles de millones USD)",
    xaxis_title="Empresa",
    xaxis_tickangle=-45,
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()
fig.write_html("inversion_streaming_vs_tradicional.html")